In [ ]:
# =============================================================================
# JHU GIS CAPSTONE (430.800) - HDBSCAN CLUSTERING OF ILLICIT AIRSTRIPS AND MINING SITES
# Author: ATD
# Date: February/March 2026 (for Spring 2026 term)
# Project Title: Geospatial Monitoring of Illegal Mining Activities and Infrastructure Networks in the Brazilian Amazon
#
# This script implements density-based clustering (HDBSCAN) and 5 km proximity analysis
# to quantify spatial-logistical relationships between detected mining deforestation
# polygons (from CNN in GEE) and illicit airstrips (reference data).
#
# Direct support for Hypothesis H2: "DBSCAN or HDBSCAN clustering will identify
# statistically significant spatial associations, with 70–80% of detected mining
# sites located within 5 km of known or detected illicit airstrips."
#
# METHODOLOGY REFERENCE (Proposal Section 5):
# - Data preprocessing in GEE → export mining polygons
# - Centroid extraction for point-based analysis (standard for polygon proximity)
# - HDBSCAN chosen over DBSCAN/K-means for handling arbitrary shapes, varying densities,
#   and noise in clandestine networks (justified in proposal with citations to
#   GeeksforGeeks 2022 and Capellan 2024).
# =============================================================================

# ----------------------------- CAVEATS AND LIMITATIONS (IMPORTANT - READ FIRST) -----------------------------
# 1. Airstrip Data: Sourced from Earthrise Media GitHub (mining-detector/data/airstrips)
#    and collaborative NYT/Intercept Brasil investigation (2022). Columns include:
#    Latitude, Longitude, "Mining activity seen up to 20km of airstrip location",
#    Airstrip ID, Original source, Attribution.
#    → This is EXTERNAL REFERENCE data (not your CNN output). Positional accuracy may vary
#      (± tens to hundreds of meters). Some points may be outdated or partially validated.
#    → Many already flag mining activity within 20 km — this may inflate proximity stats.
#
# 2. Mining Data: Google Earth Engine asset `projects/trusty-hub-437505-e5/assets/mining_cumulative_2018_2024`
#    → Polygon FeatureCollection of cumulative mining/deforestation footprints (2018–2024).
#    → We compute CENTROIDS for clustering/proximity (common practice; preserves polygon area context).
#    → This layer represents your CV/CNN-derived output or derived product. Ensure it aligns with
#      proposal temporal scope, study hotspots (Yanomami, Munduruku), and accuracy claims (H1: 80–90%).
#    → Potential mismatches: CNN detection thresholds vs. reference validation; cloud/SAR effects.
#
# 3. Spatial Considerations:
#    - UTM Zone: EPSG:32720 (Zone 20S) used as default (suitable for much of western/central Amazon
#      including parts of Yanomami). Munduruku (Pará) may better suit Zone 21S/22S — test and document.
#    - 5 km threshold is taken directly from your H2 statement. Real logistical networks may involve
#      roads, rivers, or longer distances due to terrain.
#    - Coordinate systems: All analysis in metric CRS; visualization back to WGS84 (EPSG:4326).
#
# 4. Clustering Parameters: HDBSCAN (min_cluster_size=5, min_samples=3, epsilon=5 km).
#    Sensitive to parameters — noise points (-1) expected in sparse/remote areas.
#    Compare with DBSCAN if needed for robustness.
#
# 5. Reproducibility & Deliverables:
#    - Export GPKG for ArcGIS Pro / ArcGIS Online (JHU enterprise portal — temporary access post-graduation).
#    - Outputs support map portfolio, interactive StoryMap, hotspot reports (proposal Section 6).
#    - All deliverables for reference/educational purposes only (per proposal). Cite sources rigorously.
#    - Ethical note: Do not use unvalidated outputs for operational enforcement without CENSIPAM/IBAMA review.
#
# 6. Performance: Large mining layers may be slow — filter to study area in GEE first if needed.
#    Document any filtering, CRS choices, and sensitivity tests in your final report (Chapter 4: Methods).
# =============================================================================================================

# 1. SETUP - Mount Drive and Install Packages
from google.colab import drive
drive.mount('/content/drive')

!pip install -q geopandas hdbscan scikit-learn folium matplotlib seaborn contextily shapely

import pandas as pd
import geopandas as gpd
import hdbscan
import numpy as np
import folium
from folium.plugins import HeatMap
import warnings
warnings.filterwarnings('ignore')

print("Setup complete. Proceeding with data loading...")

# 2. LOAD AIRSTrip DATA (Earthrise Media / NYT 2022)
# Update path if needed. Columns based on your provided table/screenshot:
# Latitude, Longitude, "Mining activity seen up to 20km of airstrip location", etc.
airstrip_path = '/content/drive/MyDrive/Amazon_Mining_Project/IllegalAirstripsNYT.csv'   # ←←← CONFIRM PATH

df_air = pd.read_csv(airstrip_path)

# Create GeoDataFrame (WGS84)
gdf_air = gpd.GeoDataFrame(
    df_air,
    geometry=gpd.points_from_xy(df_air['Longitude'], df_air['Latitude']),
    crs="EPSG:4326"
)

print(f"Loaded {len(gdf_air)} illicit airstrips from Earthrise Media / NYT source.")
print("Columns:", gdf_air.columns.tolist())
print(gdf_air.head(3))

# 3. LOAD MINING POLYGONS FROM GEE ASSET
# You must first EXPORT the asset from GEE as GeoJSON / GeoPackage / Shapefile
# (FeatureCollection → Export → Table to Drive or Asset).
# Recommended: Filter to Yanomami/Munduruku in GEE before export for performance.

mining_path = '/content/drive/MyDrive/Amazon_Mining_Project/amazon_basin_48px_v3.2-3.7ensemble_dissolved-0.6_2018-2024cumulative.geojson'  # ←←← UPDATE WITH YOUR EXPORT PATH

gdf_mine_poly = gpd.read_file(mining_path)

# Compute centroids (standard for polygon-to-point proximity/clustering)
gdf_mine = gdf_mine_poly.copy()
gdf_mine['centroid'] = gdf_mine_poly.geometry.centroid
gdf_mine = gdf_mine.set_geometry('centroid')

print(f"Loaded {len(gdf_mine)} mining polygons. Using centroids for analysis.")

# Optional: Filter to study hotspots if not done in GEE (example bounding box for Yanomami approx.)
# gdf_mine = gdf_mine.cx[-64:-59, 1:5]   # Adjust as needed

# 4. REPROJECT TO METRIC CRS + ANALYSIS (H2 Proximity + HDBSCAN)
# UTM Zone 20S (EPSG:32720) — suitable for much of the study area.
# Test Zone 21S (EPSG:32721) for eastern portions (Munduruku) and document choice.
gdf_air = gdf_air.to_crs("EPSG:32720")
gdf_mine = gdf_mine.to_crs("EPSG:32720")

# 4a. Simple 5 km proximity test (direct H2 metric)
buffer_air = gdf_air.buffer(5000)  # 5 km
mines_within_5km = gdf_mine.sjoin(
    gpd.GeoDataFrame(geometry=buffer_air, crs=gdf_air.crs),
    how="inner",
    predicate="intersects"
)

pct_within = (len(mines_within_5km) / len(gdf_mine)) * 100 if len(gdf_mine) > 0 else 0
print("\n=== H2 TEST RESULT ===")
print(f"{pct_within:.1f}% of mining sites (centroids) are within 5 km of an illicit airstrip.")
print("Proposal target: 70–80%. Document any deviation and potential causes (data mismatch, detection thresholds, etc.).")

# 4b. HDBSCAN Clustering on combined points
gdf_mine = gdf_mine.assign(type='mining')
gdf_air = gdf_air.assign(type='airstrip')

# Reprojecting again for clarity and robustness, ensuring consistent CRS before concatenation.
# For gdf_mine, the active geometry is 'centroid'.
gdf_mine_reproj = gdf_mine.to_crs("EPSG:32720")
# For gdf_air, the active geometry is 'geometry'.
gdf_air_reproj = gdf_air.to_crs("EPSG:32720")

# Create new GeoDataFrames for concatenation, ensuring the desired point geometry is named 'geometry'
gdf_mine_for_concat = gpd.GeoDataFrame(
    gdf_mine_reproj[['type']].copy(),
    geometry=gdf_mine_reproj.geometry,
    crs=gdf_mine_reproj.crs
)

gdf_air_for_concat = gpd.GeoDataFrame(
    gdf_air_reproj[['type']].copy(),
    geometry=gdf_air_reproj.geometry,
    crs=gdf_air_reproj.crs
)

gdf_combined = pd.concat([gdf_mine_for_concat, gdf_air_for_concat], ignore_index=True)

coords = np.column_stack((gdf_combined.geometry.x, gdf_combined.geometry.y))

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=5,
    min_samples=3,
    cluster_selection_epsilon=5000,   # 5 km core distance
    metric='euclidean'
)

gdf_combined['cluster'] = clusterer.fit_predict(coords)
gdf_combined['cluster_label'] = gdf_combined['cluster'].apply(lambda x: f'Cluster {x}' if x >= 0 else 'Noise')

# Cluster summary (key table for Results & Discussion)
cluster_summary = gdf_combined.groupby('cluster_label')['type'].value_counts().unstack(fill_value=0)
print("\nCluster Summary (Mines vs Airstrips per cluster):")
print(cluster_summary)

# 5. EXPORT DELIVERABLES (for ArcGIS Online, map portfolio, GitHub)
output_shp = '/content/drive/MyDrive/Amazon_Mining_Project/hdbscan_mine_airstrip_clusters.shp'
gdf_combined.to_file(output_shp, driver='ESRI Shapefile')
print(f"\nExported combined clusters to: {output_shp}")
print("Import this into ArcGIS Pro/Online for final layouts and StoryMap.")

# 6. INTERACTIVE VISUALIZATION (for deliverables) - WITH FIXED LEGEND
gdf_map = gdf_combined.to_crs("EPSG:4326")

# Extract x and y coordinates into separate columns for easier access
gdf_map['x'] = gdf_map.geometry.x
gdf_map['y'] = gdf_map.geometry.y

m = folium.Map(location=[-2.5, -60.0], zoom_start=6, tiles='CartoDB positron')

# Mining (red), Airstrips (blue)
for _, row in gdf_map[gdf_map['type']=='mining'].iterrows():
    folium.CircleMarker([row.geometry.y, row.geometry.x],
                        radius=3, color='red', fill=True, popup='Mining Site').add_to(m)

for _, row in gdf_map[gdf_map['type']=='airstrip'].iterrows():
    folium.CircleMarker([row.geometry.y, row.geometry.x],
                        radius=4, color='blue', fill=True, popup='Illicit Airstrip').add_to(m)

# Heatmap of clusters (excluding noise)
cluster_points = gdf_map[gdf_map['cluster'] >= 0][['y', 'x']].values
HeatMap(cluster_points, radius=15, blur=10).add_to(m)

# FIXED LEGEND - using reliable Unicode circles and spans
legend_html = '''
<div style="position: fixed;
            bottom: 50px; left: 50px; width: 190px; height: 140px;
            border:2px solid grey; z-index:9999; font-size:14px;
            background-color:white; opacity:0.95; padding: 10px; border-radius: 5px;">
    <b>Legend</b><br>
    <span style="color:#FF0000; font-size:20px;">●</span> Mining Sites<br>
    <span style="color:#0000FF; font-size:20px;">●</span> Illicit Airstrips<br>
    <span style="background:#87CEEB; opacity:0.7; display:inline-block; width:20px; height:8px; border-radius:3px; vertical-align:middle;"></span> Cluster Hotspots
</div>
'''

m.get_root().html.add_child(folium.Element(legend_html))

map_html = '/content/drive/MyDrive/Amazon_Mining_Project/mine_airstrip_interactive_map.html'
m.save(map_html)
print(f"Interactive map saved to: {map_html}")
m  # Displays in Colab

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# =============================================================================
# JHU GIS CAPSTONE (430.800) - HDBSCAN CLUSTER ANALYSIS - PROJECT_AOI
# Clipped to Yanomami + Munduruku Indigenous Territories (Tis_TerritoriosIndigenas.shp)
# Author: ATD
# Date: March 2026
# Project Title: Geospatial Monitoring of Illegal Mining Activities and Infrastructure Networks in the Brazilian Amazon
# =============================================================================

# ----------------------------- CAVEATS AND LIMITATIONS (IMPORTANT - READ FIRST) -----------------------------
# 1. Airstrip Data: MapBiomas Clandestine Airstrips (2,869 points, 1,040 Sentinel-1 validated, 2023)
#    → External reference data. Positional accuracy may vary (± tens to hundreds of meters).
#
# 2. Mining Data: Earthrise Media mining-detector repository (Amazon Mining Watch)
#    → Cumulative mining scar polygons (machine-learning ensemble detections with human validation).
#    → Exported file: amazon_basin_48px_v3.2-3.7ensemble_dissolved-0.6_2018-2024cumulative.geojson
#
# 3. AOI: Clipped to Tis_TerritoriosIndigenas.shp (Yanomami + Munduruku territories only)
#
# 4. All other caveats from the original code remain unchanged (temporal mismatch, alternative access routes, etc.).
# =============================================================================================================

# 1. SETUP
from google.colab import drive
drive.mount('/content/drive')

!pip install -q geopandas hdbscan scikit-learn folium matplotlib seaborn contextily shapely

import pandas as pd
import geopandas as gpd
import hdbscan
import numpy as np
import folium
from folium.plugins import HeatMap
import warnings
warnings.filterwarnings('ignore')

print("Setup complete. Proceeding with Project_AOI analysis...")

# 2. LOAD ALL INPUT FILES
airstrip_path = '/content/drive/MyDrive/Amazon_Mining_Project/IllegalAirstripsNYT.csv'
mining_path   = '/content/drive/MyDrive/Amazon_Mining_Project/amazon_basin_48px_v3.2-3.7ensemble_dissolved-0.6_2018-2024cumulative.geojson'
territories_path = '/content/drive/MyDrive/Amazon_Mining_Project/Tis_TerritoriosIndigenas.shp'

df_air = pd.read_csv(airstrip_path)
gdf_air = gpd.GeoDataFrame(
    df_air,
    geometry=gpd.points_from_xy(df_air['Longitude'], df_air['Latitude']),
    crs="EPSG:4326"
)

gdf_mine_poly = gpd.read_file(mining_path)
gdf_territories = gpd.read_file(territories_path)

print(f"Loaded {len(gdf_air)} airstrips, {len(gdf_mine_poly)} mining polygons, and territories shapefile.")

# 3. CLIP TO PROJECT_AOI (Yanomami + Munduruku)
gdf_mine_clipped = gpd.clip(gdf_mine_poly, gdf_territories)
gdf_air_clipped  = gpd.clip(gdf_air, gdf_territories)

print(f"After clipping to Tis_TerritoriosIndigenas.shp:")
print(f"→ Mining polygons: {len(gdf_mine_clipped)}")
print(f"→ Airstrips: {len(gdf_air_clipped)}")

# Compute centroids for mining (standard for clustering/proximity)
gdf_mine = gdf_mine_clipped.copy()
gdf_mine['centroid'] = gdf_mine_clipped.geometry.centroid
gdf_mine = gdf_mine.set_geometry('centroid')

# 4. REPROJECT + ANALYSIS (same as before)
gdf_air = gdf_air_clipped.to_crs("EPSG:32720")
gdf_mine = gdf_mine.to_crs("EPSG:32720")

# 4a. H2 Proximity Test (5 km)
buffer_air = gdf_air.buffer(5000)
mines_within_5km = gdf_mine.sjoin(
    gpd.GeoDataFrame(geometry=buffer_air, crs=gdf_air.crs),
    how="inner", predicate="intersects"
)

pct_within = (len(mines_within_5km) / len(gdf_mine)) * 100 if len(gdf_mine) > 0 else 0
print("\n=== H2 TEST RESULT (PROJECT_AOI) ===")
print(f"{pct_within:.1f}% of mining centroids are within 5 km of an illicit airstrip (clipped to Yanomami + Munduruku).")

# 4b. HDBSCAN Clustering
gdf_mine = gdf_mine.assign(type='mining')
gdf_air = gdf_air.assign(type='airstrip')

gdf_mine_for_concat = gpd.GeoDataFrame(gdf_mine[['type']].copy(), geometry=gdf_mine.geometry, crs=gdf_mine.crs)
gdf_air_for_concat  = gpd.GeoDataFrame(gdf_air[['type']].copy(),  geometry=gdf_air.geometry,  crs=gdf_air.crs)

gdf_combined = pd.concat([gdf_mine_for_concat, gdf_air_for_concat], ignore_index=True)

coords = np.column_stack((gdf_combined.geometry.x, gdf_combined.geometry.y))

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=5,
    min_samples=3,
    cluster_selection_epsilon=5000,
    metric='euclidean'
)

gdf_combined['cluster'] = clusterer.fit_predict(coords)
gdf_combined['cluster_label'] = gdf_combined['cluster'].apply(lambda x: f'Cluster {x}' if x >= 0 else 'Noise')

cluster_summary = gdf_combined.groupby('cluster_label')['type'].value_counts().unstack(fill_value=0)
print("\nCluster Summary (Project_AOI):")
print(cluster_summary)

# 5. EXPORT DELIVERABLES (Project_AOI naming)
output_shp = '/content/drive/MyDrive/Amazon_Mining_Project/hdbscan_mine_airstrip_clusters_Project_AOI.shp'
gdf_combined.to_file(output_shp, driver='ESRI Shapefile')
print(f"\nExported clipped clusters to: {output_shp}")

# 6. INTERACTIVE MAP (Project_AOI) - WITH FIXED LEGEND
gdf_map = gdf_combined.to_crs("EPSG:4326")
gdf_map['x'] = gdf_map.geometry.x
gdf_map['y'] = gdf_map.geometry.y

m = folium.Map(location=[-2.5, -60.0], zoom_start=6, tiles='CartoDB positron')

# Mining (red), Airstrips (blue)
for _, row in gdf_map[gdf_map['type']=='mining'].iterrows():
    folium.CircleMarker([row.y, row.x], radius=3, color='red', fill=True, popup='Mining Site').add_to(m)

for _, row in gdf_map[gdf_map['type']=='airstrip'].iterrows():
    folium.CircleMarker([row.y, row.x], radius=4, color='blue', fill=True, popup='Illicit Airstrip').add_to(m)

# Heatmap
cluster_points = gdf_map[gdf_map['cluster'] >= 0][['y', 'x']].values
HeatMap(cluster_points, radius=15, blur=10).add_to(m)

# FIXED LEGEND
legend_html = '''
<div style="position: fixed; bottom: 50px; left: 50px; width: 190px; height: 140px;
            border:2px solid grey; z-index:9999; font-size:14px; background-color:white;
            opacity:0.95; padding: 10px; border-radius: 5px;">
    <b>Legend (Project_AOI)</b><br>
    <span style="color:#FF0000; font-size:20px;">●</span> Mining Sites<br>
    <span style="color:#0000FF; font-size:20px;">●</span> Illicit Airstrips<br>
    <span style="background:#87CEEB; opacity:0.7; display:inline-block; width:20px; height:8px; border-radius:3px;"></span> Cluster Hotspots
</div>
'''

m.get_root().html.add_child(folium.Element(legend_html))

map_html = '/content/drive/MyDrive/Amazon_Mining_Project/mine_airstrip_interactive_map_Project_AOI.html'
m.save(map_html)
print(f"Interactive map saved to: {map_html}")
m

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Setup complete. Proceeding with Project_AOI analysis...
Loaded 1269 airstrips, 8686 mining polygons, and territories shapefile.
After clipping to Tis_TerritoriosIndigenas.shp:
→ Mining polygons: 172
→ Airstrips: 62

=== H2 TEST RESULT (PROJECT_AOI) ===
69.8% of mining centroids are within 5 km of an illicit airstrip (clipped to Yanomami + Munduruku).

Cluster Summary (Project_AOI):
type           airstrip  mining
cluster_label                  
Cluster 0             1      12
Cluster 1             0       5
Cluster 10            2       6
Cluster 11            1      11
Cluster 12            4      19
Cluster 13            8      11
Cluster 14            1      14
Cluster 2             0       8
Cluster 3             1       8
Cluster 4             5       9
Cluster 5             5       8
Cluster 6             6      12
Cluster 7            11      15
Cluste